# PCA MMSS Fly/Curve Dislocation — QDB Grid Search

Runs multiple signal configs through the full QueryDrivenBacktest engine to find optimal parameters for the PCA-weighted swap spread curve/fly strategy.

**Grid dimensions** (focused — each config ~10-16min):
- GSS smoothing halflife: [2, 3, 5]
- GSS scoring halflife: [20, 30, 60]
- z_entry / z_exit thresholds
- max holding period
- Fly-only vs fly+curve universe

In [ ]:
%load_ext autoreload
%autoreload 2

import datetime, sys, os, time, itertools
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.pylab as pylab
import QuantLib as ql

plt.style.use('ggplot')
pylab.rcParams.update({
    'legend.fontsize': 'x-large', 'figure.figsize': (18, 8),
    'axes.labelsize': 'x-large', 'axes.titlesize': 'x-large',
    'xtick.labelsize': 'large', 'ytick.labelsize': 'large',
})

sys.path.append('../../')

from BT.data_handler import TimeGrid
from BT.misc import ql_cal_date_range
from BT.query_actions import AddQueryAction, UnwindPositionsAction
from BT.query_engine import QueryDrivenBacktest
from BT.query_strategy import QueryStrategy
from BT.triggers import DateTrigger, DateTriggerRequirements
from BT.query_tearsheet import QueryBacktestTearSheet

from MDP.FixedRateBonds.FixedRateBondsMDP import FixedRateBondsMDP
from MDP.IRSwaps.IRSwapsMDP import IRSwapsMDP
from Query.FixedRateBonds.FixedRateBondQuery import FixedRateBondQuery
from Query.FixedRateBonds.FixedRateBondValue import FixedRateBondValue
from Query.FixedRateBonds.carry_roll import load_us_treasury_gc_fixing_pct
from Query.IRSwaps.IRSwapQuery import IRSwapQuery
from Query.IRSwaps.IRSwapValue import IRSwapValue

from BT.signals.tfp_swap_spread import CT_MAP
from BT.signals.irswap_pca_rv_scanner import (
    IRSwapPCARVConfig, _rolling_pca_surface, compute_fly_weights,
)
from RVUtils.plt_timeseries import make_secondary_axis_plot

## Shared Data (loaded once)

In [ ]:
BT_START = datetime.date(2021, 6, 1)
BT_END = datetime.date(2026, 5, 14)
TENORS = ['2Y', '3Y', '5Y', '7Y', '10Y', '30Y']
RISK_BPV = 100_000

# Load MMSS panel
CACHE = os.path.abspath(os.path.join(os.getcwd(), '..', '..', 'BT', 'results', 'tfp_screener', 'tfp_history.parquet'))
history = pd.read_parquet(CACHE)
history.index = pd.to_datetime(history.index)
mmss_panel = pd.DataFrame({t: history[f'mmss_{t}'] for t in TENORS if f'mmss_{t}' in history.columns}).dropna()
print(f'MMSS panel: {mmss_panel.shape}')

# PCA residuals (fixed: 520d window, 3 PCs, changes)
pca_cfg = IRSwapPCARVConfig(pca_window_days=520, pca_input='changes', n_components=3, zscore_lookback_days=150)
residuals, _, var_exp, _, _ = _rolling_pca_surface(mmss_panel, pca_cfg)
print(f'PCA done. Var explained: {var_exp.dropna().tail(60).sum(axis=1).mean():.1%}')

# PC3 fly weights (fixed — computed once from full history)
fly_weights = {}
for left, belly, right in itertools.combinations(TENORS, 3):
    rates_3 = mmss_panel[[left, belly, right]].dropna()
    if len(rates_3) < 620: continue
    try:
        w, _, _ = compute_fly_weights(rates_3, pca_cfg)
        fly_weights[(left, belly, right)] = w
    except (ValueError, np.linalg.LinAlgError):
        continue
print(f'Fly weights: {len(fly_weights)} triples')

# QDB infrastructure
CAL = ql.UnitedStates(ql.UnitedStates.GovernmentBond)
tg = TimeGrid(ql_cal_date_range(ql_cal=CAL, start=BT_START, end=BT_END))
gc = pd.Series({d: load_us_treasury_gc_fixing_pct(d) for d in sorted({ts.date() for ts in tg})}, dtype=float).sort_index().ffill().bfill()
fc = {'mode': 'gc_plus_specialness', 'gc_rate': gc / 100.0, 'leg_specialness_bps': {'outright': 10.0}, 'day_count': 'ACT/360', 'haircut': 0.0}
frb_mdp = FixedRateBondsMDP(source='USTS_FEDINVEST_WSJ_LIVE-QL')
irs_mdp = IRSwapsMDP(source='ERIS_EOD_LIVE-RL_BASIC')
print(f'QDB ready: {sum(1 for _ in TimeGrid(ql_cal_date_range(ql_cal=CAL, start=BT_START, end=BT_END)))} steps')

## QDB Runner Function

In [ ]:
def gss_zscore(series, smoothing_hl, scoring_hl):
    smoothed = series.ewm(halflife=smoothing_hl, min_periods=smoothing_hl).mean()
    mu = smoothed.ewm(halflife=scoring_hl, min_periods=scoring_hl).mean()
    sigma = smoothed.ewm(halflife=scoring_hl, min_periods=scoring_hl).std()
    return (smoothed - mu) / sigma.replace(0, np.nan)

def gen_signal(z_series, z_entry, z_exit, max_hold):
    z = z_series.values
    sig = np.zeros(len(z), dtype=np.int8)
    pos = 0; hold = 0
    for i in range(len(z)):
        v = z[i]
        if np.isnan(v): sig[i]=0; pos=0; hold=0; continue
        if pos == 0:
            if v < -z_entry: pos=+1; hold=0
            elif v > z_entry: pos=-1; hold=0
        else:
            hold += 1
            ex = hold >= max_hold
            if not ex:
                if pos==+1 and v >= -z_exit: ex=True
                elif pos==-1 and v <= z_exit: ex=True
            if ex: pos=0; hold=0
        sig[i] = pos
    return pd.Series(sig, index=z_series.index)

def extract_events(sig_series, pkg_name, legs, weights, kind):
    sig = sig_series.loc[BT_START:BT_END]
    dates = [d.date() if hasattr(d, 'date') else d for d in sig.index]
    vals = sig.values; evts = []; prev = 0; ed = None; dr = 0
    for i in range(len(vals)):
        c = int(vals[i])
        if prev == 0 and c != 0: ed = dates[i]; dr = c
        elif prev != 0 and c == 0:
            evts.append(dict(entry_date=ed, exit_date=dates[i], direction=dr, pkg=pkg_name, legs=legs, weights=weights, kind=kind, tag=f'pcafly-{pkg_name.replace("/","_")}-{ed}'))
            ed = None; dr = 0
        elif prev != 0 and c != 0 and c != prev:
            evts.append(dict(entry_date=ed, exit_date=dates[i], direction=dr, pkg=pkg_name, legs=legs, weights=weights, kind=kind, tag=f'pcafly-{pkg_name.replace("/","_")}-{ed}'))
            ed = dates[i]; dr = c
        prev = c
    if ed: evts.append(dict(entry_date=ed, exit_date=dates[-1], direction=dr, pkg=pkg_name, legs=legs, weights=weights, kind=kind, tag=f'pcafly-{pkg_name.replace("/","_")}-{ed}'))
    return evts

def run_qdb_config(label, smoothing_hl, scoring_hl, z_entry, z_exit, max_hold,
                    include_curves=True, unwind_fee_bps=0.5):
    """Run one config through full QDB. Returns metrics dict with bt and mtm."""
    # Build signals + events for all packages
    all_events = []
    
    # Flies
    for (left, belly, right), w in fly_weights.items():
        name = f'{left}/{belly}/{right}'
        s2c = w[0]*residuals[left] + w[1]*residuals[belly] + w[2]*residuals[right]
        z = gss_zscore(s2c.cumsum().dropna(), smoothing_hl, scoring_hl)
        sig = gen_signal(z, z_entry, z_exit, max_hold)
        all_events.extend(extract_events(sig, name, (left, belly, right), w, 'fly'))
    
    # Curves
    if include_curves:
        for front, back in itertools.combinations(TENORS, 2):
            name = f'{front}/{back}'
            diff = (residuals[back] - residuals[front]).cumsum()
            z = gss_zscore(diff.dropna(), smoothing_hl, scoring_hl)
            sig = gen_signal(z, z_entry, z_exit, max_hold)
            all_events.extend(extract_events(sig, name, (front, back), (-1.0, 1.0), 'curve'))
    
    if not all_events:
        return None
    
    # Build triggers
    triggers = []
    for ev in all_events:
        tag = ev['tag']; direction = ev['direction']; legs = ev['legs']; weights = ev['weights']
        entry_actions = []
        for leg_tenor, w in zip(legs, weights):
            ct = CT_MAP[leg_tenor]
            spread_dir = direction * np.sign(w)
            leg_bpv = RISK_BPV * abs(w) * spread_dir
            bq = FixedRateBondQuery(cusip=ct, value=FixedRateBondValue.NPV, structure_kwargs={'bpv': leg_bpv}, meta={'financing': fc}, tags=(tag,))
            sq = IRSwapQuery(curve='USD-SOFR-1D', tenor=ct, value=IRSwapValue.NPV, structure_kwargs={'bpv': -leg_bpv}, tags=(tag,))
            entry_actions.extend([AddQueryAction(query=bq), AddQueryAction(query=sq)])
        triggers.append(DateTrigger(DateTriggerRequirements(dates=[ev['entry_date']]), actions=entry_actions))
        triggers.append(DateTrigger(DateTriggerRequirements(dates=[ev['exit_date']]), actions=[UnwindPositionsAction(match_tag=tag, fee=unwind_fee_bps * RISK_BPV * len(legs))]))
    
    # Run
    strat = QueryStrategy(name=label, triggers=triggers, mdps={'FRB': frb_mdp, 'IRS': irs_mdp})
    bt = QueryDrivenBacktest(time_grid=tg, strategy=strat)
    t0 = time.time(); bt.run(); elapsed = time.time() - t0
    
    # Extract metrics
    mtm = pd.Series(bt.mtm_history).sort_index()
    daily = mtm.diff().dropna()
    sharpe = daily.mean() / daily.std() * np.sqrt(252) if daily.std() > 0 else 0
    max_dd = (mtm - mtm.cummax()).min()
    
    n_trades = len(all_events)
    n_win = 0; avg_hold = 0
    closed = pd.DataFrame(bt.portfolio.closed_positions_log)
    if not closed.empty:
        def _t(r):
            sq = r.get('source_query', None)
            if sq and hasattr(sq, 'tags') and sq.tags: return sq.tags[0]
            p = r.get('position', None)
            if p and hasattr(p, 'source_query') and hasattr(p.source_query, 'tags') and p.source_query.tags: return p.source_query.tags[0]
            return None
        closed['tag'] = closed.apply(_t, axis=1)
        cw = closed.dropna(subset=['tag'])
        if not cw.empty:
            tp = cw.groupby('tag').agg(pnl=('realized_pnl', 'sum'), days=('holding_period_days', 'mean'))
            n_trades = len(tp); n_win = (tp['pnl'] > 0).sum(); avg_hold = tp['days'].mean()
    
    return dict(
        label=label, sharpe=round(sharpe, 3), final_mtm=round(mtm.iloc[-1]),
        max_dd=round(max_dd), trades=n_trades,
        hit=round(n_win / n_trades, 3) if n_trades > 0 else 0,
        avg_hold=round(avg_hold, 1), elapsed=round(elapsed),
        bt=bt, mtm=mtm,
    )

## Grid Search Configs

Each config runs through the full QDB (~10-16min). Keep the grid focused.

In [ ]:
configs = [
    # Vary GSS halflife params
    ('sm2_sc20',   2, 20, 1.5, 0.25, 40, True),
    ('sm3_sc20',   3, 20, 1.5, 0.25, 40, True),
    ('sm3_sc30',   3, 30, 1.5, 0.25, 40, True),   # baseline
    ('sm3_sc60',   3, 60, 1.5, 0.25, 40, True),
    ('sm5_sc30',   5, 30, 1.5, 0.25, 40, True),
    ('sm5_sc60',   5, 60, 1.5, 0.25, 40, True),
    
    # Vary z thresholds
    ('z1.0/0.0',   3, 30, 1.0, 0.0,  40, True),
    ('z1.5/0.0',   3, 30, 1.5, 0.0,  40, True),
    ('z2.0/0.25',  3, 30, 2.0, 0.25, 40, True),
    ('z2.0/0.5',   3, 30, 2.0, 0.5,  40, True),
    
    # Vary max hold
    ('h20',        3, 30, 1.5, 0.25, 20, True),
    ('h60',        3, 30, 1.5, 0.25, 60, True),
    
    # Fly-only (no curves)
    ('fly_only',   3, 30, 1.5, 0.25, 40, False),
]

print(f'{len(configs)} configs to run')

In [ ]:
results = []
for i, (label, sm, sc, ze, zx, mh, inc_curves) in enumerate(configs):
    print(f'\n[{i+1}/{len(configs)}] {label}: sm={sm} sc={sc} z={ze}/{zx} h={mh} curves={inc_curves}', flush=True)
    r = run_qdb_config(label, sm, sc, ze, zx, mh, include_curves=inc_curves)
    if r:
        results.append(r)
        print(f'  → Sharpe={r["sharpe"]}  MTM={r["final_mtm"]:,}  trades={r["trades"]}  hold={r["avg_hold"]}d  {r["elapsed"]}s', flush=True)
    else:
        print(f'  → no trades', flush=True)

## Results

In [ ]:
summary = pd.DataFrame([{k: v for k, v in r.items() if k not in ('bt', 'mtm')} for r in results])
summary = summary.sort_values('sharpe', ascending=False)
summary

In [ ]:
# Equity curves — top 6
plot, fig, ax, ax2, legend = make_secondary_axis_plot(
    ylabel_left='Cumulative P&L', title='PCA Fly Grid Search — Top Configs',
)
for r in sorted(results, key=lambda x: -x['sharpe'])[:6]:
    plot(r['mtm'].rename(r['label']), which='left', indicators=[{'kind': 'last', 'hide': True}])
legend(show_date=True)
plt.show()

## Best Config — Tearsheet

In [ ]:
best = max(results, key=lambda r: r['sharpe'])
print(f'Best: {best["label"]}')
print(f'Sharpe={best["sharpe"]}  MTM={best["final_mtm"]:,}  MaxDD={best["max_dd"]:,}  Trades={best["trades"]}  Hit={best["hit"]:.0%}  Hold={best["avg_hold"]}d')

try:
    tearsheet = QueryBacktestTearSheet.from_backtest(best['bt'])
    plotly_fig = tearsheet.plot_plotly()
    plotly_fig.show()
except Exception as e:
    print(f'Tearsheet: {e}')

## Parameter Sensitivity

In [ ]:
# Group by parameter variation
print('GSS Smoothing HL sensitivity (scoring=30, z=1.5/0.25, h=40):')
for r in results:
    if r['label'].startswith('sm') and 'sc30' in r['label']:
        print(f'  {r["label"]:>10s}: Sharpe={r["sharpe"]:+.3f}  trades={r["trades"]}  hold={r["avg_hold"]}d')
    elif r['label'] == 'sm3_sc30':
        print(f'  {r["label"]:>10s}: Sharpe={r["sharpe"]:+.3f}  trades={r["trades"]}  hold={r["avg_hold"]}d  ← baseline')

print('\nGSS Scoring HL sensitivity (smoothing=3):')
for r in results:
    if r['label'].startswith('sm3_sc'):
        print(f'  {r["label"]:>10s}: Sharpe={r["sharpe"]:+.3f}  trades={r["trades"]}  hold={r["avg_hold"]}d')

print('\nZ-threshold sensitivity:')
for r in results:
    if r['label'].startswith('z'):
        print(f'  {r["label"]:>10s}: Sharpe={r["sharpe"]:+.3f}  trades={r["trades"]}  hold={r["avg_hold"]}d')

print('\nMax hold sensitivity:')
for r in results:
    if r['label'].startswith('h') or r['label'] == 'sm3_sc30':
        print(f'  {r["label"]:>10s}: Sharpe={r["sharpe"]:+.3f}  trades={r["trades"]}  hold={r["avg_hold"]}d')

print('\nFly-only vs fly+curve:')
for r in results:
    if r['label'] in ('sm3_sc30', 'fly_only'):
        print(f'  {r["label"]:>10s}: Sharpe={r["sharpe"]:+.3f}  trades={r["trades"]}  hold={r["avg_hold"]}d')